# Auto Update — Todas as Tabelas de Preço

Roda `atualiza_tabela_preco.ipynb` para cada tabela via **papermill**.

- Cada execução gera um notebook de output em `outputs/` com todos os prints
- Se uma tabela falhar, as demais continuam
- Instale: `pip install papermill`

In [ ]:
import papermill as pm
import time
import os
from datetime import datetime

os.makedirs('outputs', exist_ok=True)

# (nome, COD_PREVEN, EMP)
TABELAS = [
    ('CENTURY',            155, 1),
    ('PONTOVIRGULA',       355, 1),
    ('CENTURY_DOLAR',      255, 1),
    ('PONTOVIRGULA_DOLAR', 455, 1),
    ('SACCARO',             96, 1),
    ('CENTURY_WOOD',       154, 4),
    ('PV_WOOD',            354, 4),
    ('PV_WOOD_DOLAR',      454, 4),
    ('CENTURY_WOOD_DOLAR', 254, 4),
]

print(f'Tabelas a processar: {len(TABELAS)}')
for nome, cod, emp in TABELAS:
    print(f'  {nome:<25} cod_preven={cod}  emp={emp}')

In [ ]:
SEP = '=' * 60
ts_run = datetime.now().strftime('%Y%m%d_%H%M%S')
resultados = []
total_start = time.time()

for i, (nome, cod_preven, emp) in enumerate(TABELAS, 1):
    print(SEP)
    print(f'  [{i}/{len(TABELAS)}]  {nome}  (cod_preven={cod_preven}, emp={emp})')
    print(SEP, flush=True)

    output_path = f'outputs/{ts_run}_{nome}.ipynb'
    t0 = time.time()
    status = 'OK'

    try:
        pm.execute_notebook(
            'atualiza_tabela_preco_runner.ipynb',
            output_path,
            parameters={'COD_PREVEN': cod_preven, 'EMP': emp},
            progress_bar=True,
        )
    except pm.exceptions.PapermillExecutionError as e:
        status = 'ERRO'
        print(f'  ERRO em {nome}: {e.evalue}')
    except Exception as e:
        status = 'ERRO'
        print(f'  ERRO em {nome}: {e}')

    elapsed = time.time() - t0
    mins, secs = divmod(int(elapsed), 60)
    print(f'  {nome}: {status}  ({mins}m{secs:02d}s)  → {output_path}\n', flush=True)
    resultados.append((nome, cod_preven, emp, status, f'{mins}m{secs:02d}s', output_path))

total = time.time() - total_start
mins_t, secs_t = divmod(int(total), 60)
print(SEP)
print(f'  Concluído em {mins_t}m{secs_t:02d}s')
print(SEP)

In [ ]:
import pandas as pd

df_res = pd.DataFrame(resultados, columns=['tabela', 'cod_preven', 'emp', 'status', 'tempo', 'output'])
print(df_res[['tabela', 'cod_preven', 'emp', 'status', 'tempo']].to_string(index=False))